# Experiment 07: Implement Dimensionality Reduction Using PCA in Python

**Aim:** To perform dimensionality reduction using Principal Component Analysis (PCA) on the 30-feature Breast Cancer Wisconsin Diagnostic dataset, determine the component threshold for 95% variance retention, visualize 2D projections, and compare downstream classification performance.

### Dataset Description
**Dataset:** Breast Cancer Wisconsin Diagnostic Dataset (569 FNA biopsies, 30 continuous nuclear morphological features, 2 diagnostic classes: Malignant [212], Benign [357]).
- 10 morphological metrics (radius, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension) evaluated across Mean, Standard Error, and Worst (largest value).

### Theory
PCA is an unsupervised linear transformation technique that maps correlated features to an orthogonal coordinate system where principal axes align with directions of maximal data variance. For standardized matrix X, PCA computes the spectral decomposition of covariance matrix Sigma = (1/n) X^T X = V Lambda V^T. The eigenvalues lambda_k quantify variance along eigenvector v_k. Features must be centered and scaled to unit variance; otherwise high-magnitude features (e.g., area > 1000) artificially dominate.

#### 1. Load Dataset, Standardize, and Fit Full PCA Decomposition

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X, y = load_breast_cancer(return_X_y=True)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca_full = PCA().fit(X_scaled)

exp_var = pca_full.explained_variance_ratio_
cum_var = np.cumsum(exp_var)
n95 = int(np.argmax(cum_var >= 0.95)) + 1
print(f'PC1 Variance: {exp_var[0]*100:.2f}%, PC2 Variance: {exp_var[1]*100:.2f}%')
print(f'First two components capture: {cum_var[1]*100:.2f}% of total variance.')
print(f'Components required for >= 95% cumulative variance: {n95} (retains {cum_var[n95-1]*100:.2f}%)')

#### 2. Scree Plot and Cumulative Explained Variance Curve

In [ ]:
plt.figure(figsize=(9, 4))
plt.bar(range(1, 31), exp_var, alpha=0.6, align='center', label='Individual Variance')
plt.step(range(1, 31), cum_var, where='mid', color='red', label='Cumulative Variance')
plt.axhline(0.95, color='black', linestyle='--', label='95% Threshold')
plt.axvline(n95, color='darkgreen', linestyle=':', label=f'{n95} Components')
plt.xlabel('Principal Component Index')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA Scree Plot — Breast Cancer Wisconsin')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

#### 3. 2D Orthogonal Projection (PC1 vs PC2)

In [ ]:
pca_2d = PCA(n_components=2).fit_transform(X_scaled)
plt.figure(figsize=(8, 6))
plt.scatter(pca_2d[y == 0, 0], pca_2d[y == 0, 1], color='crimson', label='Malignant', alpha=0.7)
plt.scatter(pca_2d[y == 1, 0], pca_2d[y == 1, 1], color='dodgerblue', label='Benign', alpha=0.7)
plt.xlabel(f'PC1 ({exp_var[0]*100:.1f}% Variance)')
plt.ylabel(f'PC2 ({exp_var[1]*100:.1f}% Variance)')
plt.title('Breast Cancer Biopsies Projected onto PC1 and PC2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

#### 4. Controlled Downstream Classification Comparison (Full vs PCA Features)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
pipe_full = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
pipe_pca = make_pipeline(StandardScaler(), PCA(n_components=n95), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
print(f'Full 30 Features Logistic Regression Accuracy: {pipe_full.score(Xte, yte):.4f}')
print(f'{n95} Principal Components Logistic Regression Accuracy: {pipe_pca.score(Xte, yte):.4f}')

### Observations
1. The first two principal components capture 63.24% of the total 30-dimensional variance (PC1 = 44.27%, PC2 = 18.97%).
2. Exactly 10 principal components are required to surpass 95% cumulative variance (retaining 95.16%), compressing the feature space by 66.7%.
3. The 2D scatter plot demonstrates clear linear separability between malignant and benign biopsies along the PC1 axis.
4. Logistic Regression trained on all 30 features achieved 98.25% accuracy; trained on just the 10 principal components, it maintained 97.37% accuracy, losing only one test prediction while compressing the input space threefold.

### Result
PCA compressed the feature dimension from 30 down to 10 orthogonal components while retaining 95.16% variance and preserving 97.37% downstream classification accuracy. Feature standardization prevents dimensional scaling bias.